In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from catboost import CatBoostRegressor
import optuna
import warnings
warnings.filterwarnings("ignore")

X = pd.read_csv("../data/processed/X_train_engineered.csv")
y = pd.read_csv("../data/processed/y_train.csv")
X_test = pd.read_csv("../data/processed/X_test_engineered.csv")
y = y.iloc[:, 0].values if y.shape[1] == 1 else y.values.ravel()

N_FOLDS = 10
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (1458, 306)
y shape: (1458,)


In [4]:
def objective_cat(trial):
    params = {
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        "iterations": trial.suggest_int("iterations", 500, 3000),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1.0, 10.0, log=True),
        "depth": 6,
        "subsample": 0.8,
        "random_seed": 42,
        "verbose": False,
    }

    oof = np.zeros(len(X))
    for train_idx, val_idx in kf.split(X):
        X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_val = y[train_idx], y[val_idx]

        model = CatBoostRegressor(**params, early_stopping_rounds=100)
        model.fit(X_tr, y_tr, eval_set=(X_val, y_val), use_best_model=True)
        oof[val_idx] = model.predict(X_val)

    return rmse(y, oof)


study_cat = optuna.create_study(direction="minimize")
study_cat.optimize(objective_cat, n_trials=10)

print("Best params:", study_cat.best_params)
print("Best OOF RMSE:", study_cat.best_value)

[I 2026-08-28 00:26:23,436] A new study created in memory with name: no-name-864bf2c0-56a5-49ac-9b7a-80787d3da924
[I 2026-08-28 00:26:30,052] Trial 0 finished with value: 0.12389823071589826 and parameters: {'learning_rate': 0.013013466830031241, 'iterations': 536, 'l2_leaf_reg': 9.109489517697677}. Best is trial 0 with value: 0.12389823071589826.
[I 2026-08-28 00:26:43,541] Trial 1 finished with value: 0.11169784810882649 and parameters: {'learning_rate': 0.02124240772214936, 'iterations': 1381, 'l2_leaf_reg': 3.0043032553793876}. Best is trial 1 with value: 0.11169784810882649.
[I 2026-08-28 00:26:55,543] Trial 2 finished with value: 0.1113699520872228 and parameters: {'learning_rate': 0.032135119633468344, 'iterations': 1454, 'l2_leaf_reg': 3.7326999259272564}. Best is trial 2 with value: 0.1113699520872228.
[I 2026-08-28 00:27:04,303] Trial 3 finished with value: 0.11298671479944113 and parameters: {'learning_rate': 0.06791026870045655, 'iterations': 2736, 'l2_leaf_reg': 7.09538159

Best params: {'learning_rate': 0.017159747449095226, 'iterations': 2547, 'l2_leaf_reg': 2.7381480963510665}
Best OOF RMSE: 0.11087910708926411


In [5]:
best_params = study_cat.best_params
best_params.update({"depth": 6, "subsample": 0.8, "random_seed": 42, "verbose": False})

oof_cat_v2 = np.zeros(len(X))
test_cat_v2 = np.zeros(len(X_test))

for fold, (train_idx, val_idx) in enumerate(kf.split(X)):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y[train_idx], y[val_idx]

    model = CatBoostRegressor(**best_params, early_stopping_rounds=100)
    model.fit(X_tr, y_tr, eval_set=(X_val, y_val), use_best_model=True)

    oof_cat_v2[val_idx] = model.predict(X_val)
    test_cat_v2 += model.predict(X_test) / N_FOLDS

    print(f"Fold {fold+1}/{N_FOLDS} done")

final_rmse = rmse(y, oof_cat_v2)
print(f"Final tuned CatBoost OOF RMSE: {final_rmse:.5f}")

# Save for the updated ensemble
np.save("../output/oof/cat_oof_v2.npy", oof_cat_v2)
np.save("../output/oof/cat_test_v2.npy", test_cat_v2)

Fold 1/10 done
Fold 2/10 done
Fold 3/10 done
Fold 4/10 done
Fold 5/10 done
Fold 6/10 done
Fold 7/10 done
Fold 8/10 done
Fold 9/10 done
Fold 10/10 done
Final tuned CatBoost OOF RMSE: 0.11088


In [6]:
# Load the other two models' predictions from Day 7
xgb_oof = np.load("../output/oof/xgb_oof.npy")
xgb_test = np.load("../output/oof/xgb_test.npy")
lgb_oof = np.load("../output/oof/lgb_oof.npy")
lgb_test = np.load("../output/oof/lgb_test.npy")

from scipy.optimize import minimize

# Build new matrix with the tuned CatBoost
oof_matrix_v2 = np.vstack([xgb_oof, lgb_oof, oof_cat_v2]).T
test_matrix_v2 = np.vstack([xgb_test, lgb_test, test_cat_v2]).T
model_names_v2 = ["xgboost", "lightgbm", "catboost_tuned"]

n_models = oof_matrix_v2.shape[1]

def weighted_rmse(weights, oof_matrix, y):
    blend = oof_matrix @ weights
    return rmse(y, blend)

init_weights = np.array([1 / n_models] * n_models)
constraints = ({"type": "eq", "fun": lambda w: np.sum(w) - 1})
bounds = [(0, 1)] * n_models

result_v2 = minimize(
    weighted_rmse, init_weights, args=(oof_matrix_v2, y),
    method="SLSQP", bounds=bounds, constraints=constraints
)

best_weights_v2 = result_v2.x
print("Optimal weights (v2):")
for name, w in zip(model_names_v2, best_weights_v2):
    print(f"  {name}: {w:.4f}")

ensemble_oof_rmse_v2 = weighted_rmse(best_weights_v2, oof_matrix_v2, y)
print(f"New ensemble OOF RMSE: {ensemble_oof_rmse_v2:.5f}")
print(f"Previous ensemble OOF RMSE (Day 8): 0.11133")

Optimal weights (v2):
  xgboost: 0.1914
  lightgbm: 0.0000
  catboost_tuned: 0.8086
New ensemble OOF RMSE: 0.11059
Previous ensemble OOF RMSE (Day 8): 0.11133


In [7]:
test_blend_log_v2 = test_matrix_v2 @ best_weights_v2
test_blend_price_v2 = np.expm1(test_blend_log_v2)

test_ids = pd.read_csv("../data/test.csv")["Id"].values

submission_v2 = pd.DataFrame({
    "Id": test_ids,
    "SalePrice": test_blend_price_v2
})
submission_v2.to_csv("../output/submission_v2.csv", index=False)
submission_v2.head()

,Id,SalePrice
0,1461,126527.102373
1,1462,162328.283050
2,1463,183730.688233
3,1464,190756.125384
4,1465,184124.615066


In [8]:
sample_sub = pd.read_csv("../data/sample_submission.csv")

checks = {
    "Row count is 1459": len(submission_v2) == 1459,
    "All prices are non-negative": (submission_v2["SalePrice"] >= 0).all(),
    "Id order matches sample_submission": (submission_v2["Id"].values == sample_sub["Id"].values).all(),
}

for check_name, passed in checks.items():
    status = "PASSED" if passed else "FAILED"
    print(f"{check_name}: {status}")

assert all(checks.values()), "Submission file has issues, please check!"

Row count is 1459: PASSED
All prices are non-negative: PASSED
Id order matches sample_submission: PASSED
